In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

df = pd.read_csv("window_feature_mean_by_condition_new.csv")

features = ["P", "M", "Std", "SLTA", "K", "S"]

grouped = df.groupby(["feature", "window_size", "snr_db"], as_index=False)["pearson_mean"].mean()
grouped.rename(columns={"pearson_mean": "avg_pearson"}, inplace=True)

pivot_dict = {}
for feat in features:
    sub = grouped[grouped["feature"] == feat]
    pivot = sub.pivot(index="snr_db", columns="window_size", values="avg_pearson")
    pivot_dict[feat] = pivot
    print(f"\n=== Feature {feat}：mean Pearson across windows at each SNR ===")
    print(pivot.round(4))

global_perf = grouped.groupby(["feature", "window_size"], as_index=False)["avg_pearson"].mean()
global_perf.rename(columns={"avg_pearson": "global_avg"}, inplace=True)
global_perf["rank"] = global_perf.groupby("feature")["global_avg"].rank(ascending=False, method="min")
print("\n=== global average performance ranking across all features（windows 7~21） ===")
print(global_perf.sort_values(["feature", "rank"]).round(4).to_string(index=False))

sns.set_style("ticks")
plt.rcParams["font.sans-serif"] = ["Times New Roman", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

all_values = np.concatenate([pivot_dict[f].values.ravel() for f in features])
global_vmin = all_values.min()
global_vmax = all_values.max()

fig, axes = plt.subplots(
    nrows=2, ncols=3,
    figsize=(16, 9),
    dpi=150,
    sharex=True,
    sharey=True
)
axes = axes.flatten()

sublist = "abcdef"

for i, feat in enumerate(features):
    ax = axes[i]
    pivot = pivot_dict[feat]

    sns.heatmap(
        pivot,
        ax=ax,
        cmap="YlOrRd",
        annot=False,
        cbar=False,
        linewidths=0.0,

        vmin=0.0,

        vmax=1.0,
        xticklabels=True,
        yticklabels=True
    )

    ax.text(0.03, 0.93, f"({sublist[i]})",
            transform=ax.transAxes,
            fontsize=12,
            color='black')
    ax.text(0.91, 0.93, f"{feat}",
            transform=ax.transAxes,
            fontsize=12,
            color='black')

    if i % 3 == 0:
        ax.set_ylabel("SNR (dB)", fontsize=10)
    else:
        ax.set_ylabel("")
    if i >= 3:
        ax.set_xlabel("Window Size", fontsize=10)
    else:
        ax.set_xlabel("")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
mappable = axes[0].collections[0]
fig.colorbar(
    mappable,
    cax=cbar_ax,
    label="Pc",

)

plt.subplots_adjust(right=0.9, wspace=0.04, hspace=0.08)

plt.savefig("./pearson/heatmap_6features_shared_cbar.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()


### spearman

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

df = pd.read_csv("window_feature_mean_by_condition_new.csv")

features = ["P", "M", "Std", "SLTA", "K", "S"]

grouped = df.groupby(["feature", "window_size", "snr_db"], as_index=False)["spearman_mean"].mean()
grouped.rename(columns={"spearman_mean": "avg_spearman"}, inplace=True)

pivot_dict = {}
for feat in features:
    sub = grouped[grouped["feature"] == feat]
    pivot = sub.pivot(index="snr_db", columns="window_size", values="avg_spearman")
    pivot_dict[feat] = pivot
    print(f"\n=== Feature {feat}：mean Spearman across windows at each SNR ===")
    print(pivot.round(4))

global_perf = grouped.groupby(["feature", "window_size"], as_index=False)["avg_spearman"].mean()
global_perf.rename(columns={"avg_spearman": "global_avg"}, inplace=True)
global_perf["rank"] = global_perf.groupby("feature")["global_avg"].rank(ascending=False, method="min")
print("\n=== global average performance ranking across all features ===")
print(global_perf.sort_values(["feature", "rank"]).round(4).to_string(index=False))

plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

sns.set_style("ticks")
plt.rcParams["font.sans-serif"] = ["SimHei", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

all_values = np.concatenate([pivot_dict[f].values.ravel() for f in features])
global_vmin = all_values.min()
global_vmax = all_values.max()

fig, axes = plt.subplots(
    nrows=2, ncols=3,
    figsize=(16, 9),
    dpi=150,
    sharex=True,
    sharey=True
)
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    pivot = pivot_dict[feat]

    sns.heatmap(
        pivot,
        ax=ax,
        cmap="mako",
        annot=False,
        cbar=False,
        linewidths=0.0,
        vmin=global_vmin,
        vmax=global_vmax+0.10,
        xticklabels=True,
        yticklabels=True
    )

    ax.set_title(f"Feature {feat}", fontsize=12, pad=8)

    if i % 3 == 0:
        ax.set_ylabel("SNR (dB)", fontsize=10)
    else:
        ax.set_ylabel("")
    if i >= 3:
        ax.set_xlabel("Window size", fontsize=10)
    else:
        ax.set_xlabel("")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
mappable = axes[0].collections[0]
fig.colorbar(
    mappable,
    cax=cbar_ax,
    label="Frequency-averaged Spearman correlation",
    orientation="vertical"
)

fig.suptitle(
    "Spearman correlation heatmap for six features across windows and SNR values",
    fontsize=14,
    y=0.96
)

plt.subplots_adjust(right=0.9, wspace=0.12, hspace=0.2)

plt.savefig("./spearman/heatmap_6features_shared_cbar.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()


### MI

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

df = pd.read_csv("window_feature_mean_by_condition_new.csv")

features = ["P", "M", "Std", "SLTA", "K", "S"]

grouped = df.groupby(["feature", "window_size", "snr_db"], as_index=False)["mi_mean"].mean()
grouped.rename(columns={"mi_mean": "avg_mi"}, inplace=True)

pivot_dict = {}
for feat in features:
    sub = grouped[grouped["feature"] == feat]
    pivot = sub.pivot(index="snr_db", columns="window_size", values="avg_mi")
    pivot_dict[feat] = pivot
    print(f"\n=== Feature {feat}：mean mutual information across windows at each SNR ===")
    print(pivot.round(4))

global_perf = grouped.groupby(["feature", "window_size"], as_index=False)["avg_mi"].mean()
global_perf.rename(columns={"avg_mi": "global_avg"}, inplace=True)
global_perf["rank"] = global_perf.groupby("feature")["global_avg"].rank(ascending=False, method="min")
print("\n=== global average performance ranking across all features ===")
print(global_perf.sort_values(["feature", "rank"]).round(4).to_string(index=False))

sns.set_style("ticks")
plt.rcParams["font.sans-serif"] = ["SimHei", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

all_values = np.concatenate([pivot_dict[f].values.ravel() for f in features])
global_vmin = all_values.min()
global_vmax = all_values.max()

fig, axes = plt.subplots(
    nrows=2, ncols=3,
    figsize=(16, 9),
    dpi=150,
    sharex=True,
    sharey=True
)
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    pivot = pivot_dict[feat]

    sns.heatmap(
        pivot,
        ax=ax,
        cmap="PuBu",
        annot=False,
        cbar=False,
        linewidths=0.0,
        vmin=global_vmin,
        vmax=global_vmax+0.05,
        xticklabels=True,
        yticklabels=True
    )

    ax.set_title(f"Feature {feat}", fontsize=12, pad=8)

    if i % 3 == 0:
        ax.set_ylabel("SNR (dB)", fontsize=10)
    else:
        ax.set_ylabel("")
    if i >= 3:
        ax.set_xlabel("Window size", fontsize=10)
    else:
        ax.set_xlabel("")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
mappable = axes[0].collections[0]
fig.colorbar(
    mappable,
    cax=cbar_ax,
    label="Frequency-averaged mutual information",
    orientation="vertical"
)

fig.suptitle(
    "Mutual-information heatmap for six features across windows and SNR values",
    fontsize=14,
    y=0.96
)

plt.subplots_adjust(right=0.9, wspace=0.12, hspace=0.2)

plt.savefig("./mi/heatmap_6features_shared_cbar.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

snrs_to_plot = [-10, -8, -6]
df_plot = df[df['snr_db'].isin(snrs_to_plot)].copy()

sns.set_theme(style="whitegrid", font_scale=1.1)

g = sns.FacetGrid(df_plot, col="snr_db", col_wrap=3, height=4, aspect=1.2,
                  hue_kws={"marker": "o"})

g.map_dataframe(sns.lineplot, x="window_size", y="mi_mean",
                hue="feature", style="feature", markers=True, dashes=False, linewidth=2, errorbar=None)

for ax in g.axes.flat:
    ax.axvspan(9, 13, color='green', alpha=0.15, label='Optimal Window (11-13)')
    ax.set_xticks(range(7, 36, 2))
    ax.legend(loc='lower right', fontsize=9)

g.set_axis_labels("Window Size", "Spearman Correlation")
g.set_titles("SNR = {col_name} dB")
plt.subplots_adjust(top=0.9)
g.fig.suptitle("Feature Correlation vs. Window Size (Spearman)", fontsize=14, fontweight='bold')
plt.show()


In [ ]:
metrics = ['mi_mean', 'pearson_mean', 'spearman_mean']
snrs = [-10, -8, -6]

with pd.ExcelWriter('aggregated_tables.xlsx') as writer:
    for metric in metrics:
        for snr in snrs:

            df_sub = df_agg[df_agg['snr_db'] == snr]

            table = df_sub.pivot_table(index='window_size',
                                       columns='feature',
                                       values=metric,
                                       aggfunc='mean')

            feature_order = ['K', 'M', 'P', 'SLTA', 'Std', 'S']

            table = table.reindex(columns=feature_order)

            table = table.sort_index()

            sheet_name = f"{metric}_SNR_{snr}"

            table.to_excel(writer, sheet_name=sheet_name)

# print(pd.read_excel('aggregated_tables.xlsx', sheet_name='mi_mean_SNR_-10'))
